# Implementation of Lambert Liu

### Imports and config dicts

In [3]:
import numpy as np 
import polars as pl 
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from numba import njit  
import math 
from collections import namedtuple

### Loading and saving data 

In [ ]:
df = pl.scan_parquet(f'{data_path}/input/{input_data}.parquet')

train_test_dict = get_train_test_split()
bin_metric_dict = get_bin_metrics()
bin_metric_dict = add_training_denom(bin_metric_dict)


### Getting helper dictionaries

In [ ]:
# Uses the above functions to get 2 dictionaries 
# train_test_dict with the fine bins index upon which we make the train test split
# bin metric dict which calculates metric such as fine bins per day and fine bins used in the training estimate


### Preprocessing data

This involves:

    - Turning the source_user@domain column into a user_id to make it more lightweight
    - Making the fine bin column and getting counts in each fine bin
    - Making the course bin column
    - Getting the fine bin within coarse bin position

In [7]:
# Doing initial manipulations on the data including replacing source_user@domain with user_id and constructing course and fine bins
user_counts = create_counts_data(df)
user_counts, user_mapping = create_user_to_id_mapping(user_counts, mapping_file_name='source_users_to_id_mapping')
user_counts = create_coarse_bins(user_counts)
user_interactions = create_first_last_interaction_arrays(user_counts=user_counts)

### Getting inital parameter estimates and interpolation weights

In [ ]:
def get_period_sums(user_counts, period_start, period_end):
    ''' 
    Gets the sums of a df in the given period, the 

    This data is used to get the initial parameter estimates
    It is also used to create the grids for clustering
    '''

    period_df = user_counts.filter((pl.col('fine_bin_id') >= period_start)
                                    & (pl.col('fine_bin_id') < period_end))

    period_df = period_df.with_columns(count_2 = pl.col('count') ** 2)
    period_df = period_df.group_by(['user_id', 'coarse_bin_id']).agg(pl.sum('count').alias('sum_cnt'), 
                                                                    pl.sum('count_2').alias('sum_cnt_2'),
                                                                    pl.len().alias('n_bins'))
    
    return period_df 

def get_fine_bins_per_cb(period_start, period_end, bin_metric_dict=bin_metric_dict):
    ''' 
    Returns the number of fine bins per coarse bin in the period
    '''
    return ((period_end - period_start)//bin_metric_dict['fine_bins_per_week']) * bin_metric_dict['fine_bins_per_coarse_bin']


def init_grid_NB(user_counts, n_users, coarse_bins_per_week, period_start, period_end):
    ''' 
    Creates the u and v init grids from the training data
    '''
    # Getting the sum of counts and sum of counts squared needed for mean and variance calculations
    train_df = get_period_sums(user_counts, period_start, period_end)
    fb_per_cb = get_fine_bins_per_cb(period_start, period_end)
    
    # Init a grid of parmeters to use
    u_init = np.zeros((n_users, coarse_bins_per_week), dtype='float64')
    v_init = np.zeros((n_users, coarse_bins_per_week), dtype='float64')

    # Extrating the entries to assign and assigning them to the df
    entries_to_assign = train_df.select(['user_id', 'coarse_bin_id']).to_numpy()

    u_init[entries_to_assign[:,0], entries_to_assign[:,1]] = train_df['sum_cnt'].to_numpy() / fb_per_cb
    v_init[entries_to_assign[:,0], entries_to_assign[:,1]] = (train_df['sum_cnt_2'].to_numpy() - 
                                                            ((train_df['sum_cnt'] **2) / fb_per_cb))/(fb_per_cb - 1)

    # Capping the min values of u_init and v_init
    u_init = np.maximum(u_init, config_dict['mean_min'])
    v_init = np.maximum(v_init, config_dict['var_min'])

    return u_init, v_init

def init_grid_hurdle(user_counts, n_users, coarse_bins_per_week, period_start, period_end):

    # Getting the sum of counts and sum of counts squared needed for mean and variance calculations
    train_df = get_period_sums(user_counts, period_start, period_end)
    fb_per_cb = get_fine_bins_per_cb(period_start, period_end)
    
    # Init a grid of parmeters to use
    u_init = np.zeros((n_users, coarse_bins_per_week), dtype='float64')
    v_init = np.zeros((n_users, coarse_bins_per_week), dtype='float64')
    p_init = np.zeros((n_users, coarse_bins_per_week), dtype='float64')

    # Extrating the entries to assign and assigning them to the df
    entries_to_assign = train_df.select(['user_id', 'coarse_bin_id']).to_numpy()

    # Getting columns as numpy arrays
    n_bins = train_df['n_bins'].to_numpy()
    sum_cnt = train_df['sum_cnt'].to_numpy()
    sum_cnt_2 = train_df['sum_cnt_2'].to_numpy()

    # Creating a mask we use to assign values
    mean_mask = n_bins > 0
    var_mask = n_bins > 1

    p_init[entries_to_assign[:,0], entries_to_assign[:,1]] = n_bins / fb_per_cb
    u_init[entries_to_assign[mean_mask, 0], entries_to_assign[mean_mask, 1]] = sum_cnt[mean_mask] / n_bins[mean_mask]
    v_init[entries_to_assign[var_mask, 0], entries_to_assign[var_mask, 1]] = ((sum_cnt_2[var_mask]- ((sum_cnt[var_mask] ** 2) / n_bins[var_mask]))
                                                                                / (n_bins[var_mask] - 1))

    # Capping the min values of u_init and v_init
    u_init = np.maximum(u_init, config_dict['mean_min'])
    v_init = np.maximum(v_init, config_dict['var_min'])
    p_init = np.minimum(np.maximum(p_init, config_dict['p_min']), config_dict['p_max'])

    return u_init, v_init, p_init

def init_n_counts_grid(user_counts, n_users, coarse_bins_per_week, period_start, period_end):
    '''
    Counts how many counts were observed in the period needed for the smoothing equation that relies on n counts
    '''

    train_df = get_period_sums(user_counts, period_start, period_end)

    # Init a grid and get entries to assign and assigning the number of counts
    n_counts = np.zeros((n_users, coarse_bins_per_week), dtype='float64')
    entries_to_assign = train_df.select(['user_id', 'coarse_bin_id']).to_numpy()
    n_counts[entries_to_assign[:, 0], entries_to_assign[:, 1]] = train_df['sum_cnt'].to_numpy()

    return n_counts

def get_degen_mask(user_counts, n_users, n_coarse_bins, config_dict, train_test_dict):
    ''' 
    Creates a grid of user x coarse bin combinations that have less than `config_dict.degen_threshold` 
    counts in them. These bins are excluded from scoring.
    '''
    # Get the number of counts for each user in the period
    train_df = get_period_sums(user_counts, train_test_dict['train_start'], train_test_dict['burn_in_end'])

    # Init the output mask with all bins as degen
    degen_mask = np.ones((n_users, n_coarse_bins), dtype='bool')

    # Finding which rows are not degen and setting the mask to false
    non_degen_rows = train_df['n_bins'].to_numpy() >= config_dict['degen_threshold']
    entries_to_assign = train_df.select(['user_id', 'coarse_bin_id']).to_numpy()
    degen_mask[entries_to_assign[non_degen_rows, 0], entries_to_assign[non_degen_rows, 1]] = False

    return degen_mask

In [ ]:
# Creatung the inital grids and saving the outputs
u_init, v_init = init_grid_NB(user_counts, n_users = user_mapping.shape[0], coarse_bins_per_week = bin_metric_dict['coarse_bins_per_week'], 
                              period_start=train_test_dict['train_start'], period_end=train_test_dict['train_end'])

u_pos_init, v_pos_init, p_init = init_grid_hurdle(user_counts, n_users = user_mapping.shape[0], coarse_bins_per_week = bin_metric_dict['coarse_bins_per_week'], 
                                                    period_start=train_test_dict['train_start'], period_end=train_test_dict['train_end'])

# Creating the grids used for clustering
u_clustering, v_clustering = init_grid_NB(user_counts, n_users = user_mapping.shape[0], coarse_bins_per_week = bin_metric_dict['coarse_bins_per_week'], 
                              period_start=train_test_dict['train_start'], period_end=train_test_dict['burn_in_end'])

u_pos_clustering, v_pos_clustering, p_pos_clustering = init_grid_hurdle(user_counts, n_users = user_mapping.shape[0], coarse_bins_per_week = bin_metric_dict['coarse_bins_per_week'], 
                                                    period_start=train_test_dict['train_start'], period_end=train_test_dict['burn_in_end'])

# Creating initial n_counts_grid
n_counts_init = init_n_counts_grid(user_counts, n_users=user_mapping.shape[0], coarse_bins_per_week=bin_metric_dict['coarse_bins_per_week'], period_start=train_test_dict['train_start'], period_end=train_test_dict['burn_in_end'])

# Getting the degen mask
degen_mask = get_degen_mask(user_counts, n_users=user_mapping.shape[0], n_coarse_bins=bin_metric_dict['coarse_bins_per_week'], config_dict=config_dict, train_test_dict=train_test_dict)

In [ ]:
# Creating degen metrics TODO (eventually move this to metrics section)
degen_bins_per_user = user_mapping.with_columns(pl.Series('n_degen_bins', degen_mask.astype('int64').sum(axis=1)))
store_data(degen_bins_per_user, 'degen_bins_per_user')

### Getting interpolation weights

In [10]:
def get_interpolation_weights(bin_metric_dict=bin_metric_dict):
    '''
    A function that returns the weights we apply when interpolating.
    To understand the computation steps see pages 11 and 12 of lambert liu
    returns:
        weights a numpy array which will be applied as w-1 U-1 + w0 U0 + w1 U1
        the weights array has a row for every fine bin in the coarse bin and 3 columns where each column is the weight being applies to Uis
    '''

    # Getting the m (fine bin number, q and r (defined in lambert liu))
    M = bin_metric_dict['fine_bins_per_coarse_bin']
    m = np.arange(1, M+1)
    q = (m - 1)/M
    r = m/M

    # Getting the two terms used in all calculations
    term_1 = r**2 + r*q + q**2
    term_2 = r + q

    # Computing the weights using the lambert and liu formula
    # these come from rearranging the fomula at the top of page 12 for U-1 U0 and U1
    weights = np.zeros((M, 3))

    weights[:, 0] = term_1/6 - term_2/2 + 1/3
    weights[:, 1] = -term_1/3 + term_2/2 + 5/6
    weights[:, 2] = term_1/ 6 - 1/6

    return weights

In [11]:
# Saving the outputs of this section
interpolation_weights = get_interpolation_weights()

### Making clustered model

#### Clustering methods

- Cell 1 matrix to cluster construction
- Cell 2 Clustering method choice function
- Cell 3 Cluster summary statistics helpers

In [ ]:
#### Creating functions that return the correct vector for clustering

# Matrix construction functions
def make_u_matrix(u, v, p): return u

def make_log_u_matrix(u, v, p): return np.log(u)

def make_v_matrix(u, v, p): return v

def make_normalised_u_clustering_matrix(u, v, p): return u / u.sum(axis=1)[:, None]


# Wrapper function
def make_clustering_matrix(u, v, p, config_dict=config_dict): 
    ''' 
    Constructs the matrix used for clustering. 
    The construction used depends on `clustering_matrix_name` found within `config_dict`
    '''
    if config_dict['clustering_matrix_name'] == 'u':
        return make_u_matrix(u, v, p)
    elif config_dict['clustering_matrix_name'] == 'log_u':
        return make_log_u_matrix(u, v, p)
    elif config_dict['clustering_matrix_name'] == 'v':
        return make_v_matrix(u, v, p)
    elif config_dict['clustering_matrix_name'] == 'normalised_u':
        return make_normalised_u_clustering_matrix(u, v, p)
    else: 
        raise ValueError("config_dict['clustering_matrix_name'] invalid")
    



In [13]:
### Creating clusters and then initialising a cluster model

def get_k_means_assignments(k, random_state, matrix_to_cluster):
    '''
    Performs k means clustering on a numpy array and returns a vector of cluster assignments
    '''

    k_means_model = KMeans(n_clusters=k, random_state=random_state)
    clusters = k_means_model.fit_predict(matrix_to_cluster).astype(np.int64)

    return clusters, k_means_model

def get_cluster_assignments(cluster_param, matrix_to_cluster, config_dict=config_dict):
    ''' 
    Generic clustering runner that can be scaled to include multiple algorithms
    '''
    if config_dict['clustering_method'] == 'k_means':
        return get_k_means_assignments(k=cluster_param, random_state=config_dict['seed'], matrix_to_cluster=matrix_to_cluster)
    else:
        raise ValueError('Clustering method not Reckognised')

In [ ]:
def get_centroid_distance(cluster_centres):
    ''' 
    For each cluster computes l2 distances and returns:
        - Average distance to other clusters
        - Min distance to other clusters
    '''
    output = {}
    for i in range(cluster_centres.shape[0]):
        cluster_distances = []
        for j in range(cluster_centres.shape[0]):
            
            # Calculate l2 distance
            if i != j:
                cluster_distances.append(np.sqrt(np.sum((cluster_centres[i] - cluster_centres[j]) ** 2)))

        # Get outputs for cluster
        if len(cluster_distances) == 0:
            output[i] = {'nearest_centroid_dist' : np.nan, 'avg_centroid_dist' : np.nan}
        else:
            output[i] = {'nearest_centroid_dist' : min(cluster_distances), 'avg_centroid_dist' : sum(cluster_distances)/len(cluster_distances)}

    return output

def create_cluster_summary_df(model, user_mapping=user_mapping, config_dict=config_dict):
    ''' 
    Creates an output df which summarises cluster quality metrics
    '''

    # Getting the number of clusters
    if config_dict['clustering_method'] == 'k_means':
        n_clusters=model['cluster_param']
    else:
        raise ValueError('Clustering method not Reckognised')
    
    output = []
    cluster_centroid_dict = get_centroid_distance(model['cluster_centres'])

    for cluster_id in range(n_clusters):

        users_in_cluster = user_mapping.filter(pl.col('user_id').is_in(np.where(model['cluster_assignments'] == cluster_id)[0]))
        n_users_in_cluster = users_in_cluster.shape[0]
        human_users_in_cluster = users_in_cluster.filter(pl.col('source_user_type') == 'human').shape[0]
        machine_users_in_cluster = users_in_cluster.filter(pl.col('source_user_type') == 'machine').shape[0]

        output.append({
            'cluster_id' : cluster_id,
            'cluster_param' : model['cluster_param'],
            'seed' : model['seed'],
            'clustering_matrix_name' : model['clustering_matrix_name'],

            # Cluster content metrics
            'n_users' : n_users_in_cluster,
            'n_humans' : human_users_in_cluster,
            'n_machine_users' : machine_users_in_cluster,

            # Performance metrics
            'inertia' : model['cluster_inertia'],
            'nearest_centroid_dist' : cluster_centroid_dict[cluster_id]['nearest_centroid_dist'],
            'avg_centroid_dist' : cluster_centroid_dict[cluster_id]['avg_centroid_dist'],
        })

    return pl.DataFrame(output)


In [ ]:
## Clustering model and getting cluster means helper 
#! NOTE also we should probably cluster the data based on the parameters for the full training data not just for the first week of train

def get_param_cluster_mean(cluster_groups, param_grid):
    ''' 
    Helper function which calculates the cluster param given a param grid (u, v or p)
    '''
    n_users, n_coarse_bins = param_grid.shape 
    n_clusters = cluster_groups.max() + 1

    # Init mean vectors
    cluster_mean = np.zeros((n_clusters, n_coarse_bins), dtype='float64')
    users_per_cluster = np.zeros(n_clusters, dtype='float64')

    # Summing u or v or p contributions in each cluster
    # Extract the cluster assignment for each user and then add their parameters to each bin
    for user_id in range(n_users):
        cluster_assignment = int(cluster_groups[user_id])
        cluster_mean[cluster_assignment, :] += param_grid[user_id, :]
        users_per_cluster[cluster_assignment] += 1

    # Dividing through to get the averages in each cluster
    for cluster_assignment in range(n_clusters):
        if users_per_cluster[cluster_assignment] > 0:
            cluster_mean[cluster_assignment, :] /= users_per_cluster[cluster_assignment]

    return cluster_mean

def get_cluster_means(cluster_groups, u_init, v_init, p_init):
    ''' 
    Calculates mean u and v and p values for each cluster group and each time bin
    Used within the get clustering model function
    Args:
        cluster_groups a n_users length vector of cluster assignments
        u_init : the calculated vector of parameter means (or non zero mean if a hurdle model)
        v_init : the calculated vector of initial parameter variances (or non zero variances if hurdle model)
        v_init : the calculated vector of initial parameter activation probs (if a hurdle model)
    '''
    return get_param_cluster_mean(cluster_groups, u_init), get_param_cluster_mean(cluster_groups, v_init), get_param_cluster_mean(cluster_groups, p_init)

def make_cluster_model(cluster_param, config_dict, u_init, v_init, p_init=None):
    ''' 
    Creates the clustering model dictionary 
    '''

    # Init p and matrix to cluster
    if p_init is None:
        p_init = np.zeros_like(u_init)
    matrix_to_cluster = make_clustering_matrix(u_init, v_init, p_init, config_dict=config_dict)

    # Global pooling defaults
    if cluster_param == 1:
        cluster_assignments = np.zeros(u_init.shape[0], dtype=np.int64)
        cluster_centres = matrix_to_cluster.mean(axis=0, keepdims=True)
        # Dont care about inertia for 1 cluster
        cluster_inertia = 0

    # Getting cluster assignments and cluster centres
    else:
        cluster_assignments, clustering_model = get_cluster_assignments(cluster_param=cluster_param, matrix_to_cluster=matrix_to_cluster, config_dict=config_dict)
        cluster_centres = clustering_model.cluster_centers_
        cluster_inertia = clustering_model.inertia_

    # Get mean cluster values
    cluster_mean_u, cluster_mean_v, cluster_mean_p = get_cluster_means(cluster_groups=cluster_assignments, u_init=u_init, v_init=v_init, p_init=p_init)
    

    output = {
        # Clustering configs
        'name' : f"{config_dict['clustering_method']}",
        'clustering_matrix_name' : config_dict['clustering_matrix_name'],
        'seed' : config_dict['seed'],
        'cluster_param' : cluster_param,

        # Identified values
        'cluster_mean_u' : cluster_mean_u,
        'cluster_mean_v' : cluster_mean_v,
        'cluster_mean_p' : cluster_mean_p,
        'cluster_assignments' : cluster_assignments,
        
        # Cluster quality metrics
        'cluster_inertia' : cluster_inertia, 
        'cluster_centres' : cluster_centres,
        }

    return output 

In [ ]:
# Getting the clustering model summarising cluster performance and storing the df
# running the lambert liu runner
def get_ll_param_grids(config_dict):
    '''
    If hurdle model creates a grid of 0s for p else just returns grids
    '''
    if config_dict['hurdle_model']:
        return u_pos_init, v_pos_init, p_init, u_pos_clustering, v_pos_clustering, p_pos_clustering
    else:
        return u_init, v_init, np.zeros_like(u_init), u_clustering, v_clustering, np.zeros_like(u_clustering)
    

_, _, _, u_cluster, v_cluster, p_cluster = get_ll_param_grids(config_dict)
clustering_model = make_cluster_model(cluster_param=config_dict['cluster_param'], config_dict=config_dict, u_init=u_cluster, v_init=v_cluster, p_init=p_cluster)

summary_df = create_cluster_summary_df(clustering_model)
store_data(summary_df, 'clustering_algorithm_performance', csv=True)

Number of clusters identified : 4


### Storing data and converting to nt for final runner

- Eventually will split the notebook here into preprocessing and the rest so we just load the data and ll

In [17]:
def dictionary_to_named_tuple_class(name : str, dictionary : dict):
    ''' 
    Converts a dict to a named tuple with the given name.
    Can be used downstream in the njit functions
    '''
    if name in globals():
        raise ValueError(f'Named tuple: {name} already exists')
    else:
        return namedtuple(name, dictionary.keys())
        
    
def df_to_nt(name, df):
    ''' 
    Converts a dataframe to a Namedtuple of numpy arrays for use in the numba runner
    '''
    table_dict = {col : df[col].to_numpy().astype('int64') for col in df.columns}

    return dictionary_to_named_tuple_class(name, table_dict)(**table_dict)

In [ ]:
# Storing data needed for the numba runner
user_counts = user_counts.select(['user_id', 'fine_bin_id', 'count'])
store_data(user_interactions, 'user_interactions')
store_data(user_counts, 'user_counts')
store_data(interpolation_weights, filename='interpolation_weights')
store_data(u_init, 'u_init')
store_data(v_init, 'v_init')
store_data(p_init, 'p_init')
store_data(u_pos_init, 'u_pos_init')
store_data(v_pos_init, 'v_pos_init')
store_data(n_counts_init, 'n_counts_init')

##### Creating NT LL args

In [19]:
# Converting the dfs to nt of numpy arrays to be used for the final numba runner
user_interactions_nt = df_to_nt('user_interactions_nt', user_interactions)
user_counts_nt = df_to_nt('user_counts_nt', user_counts)

In [ ]:
# Creating dictionaries of names of outputs and oder they appear in
output_names = ['n_bins_scored', 'non_degen_ll_sum','non_degen_smoothed_ll_sum']
model_names = ['raw_model_calib_index', 'smoothed_model_calib_index']

output_idx_dict = {name : idx for idx, name in enumerate(output_names)}
model_idx_dict = {name : idx for idx, name in enumerate(model_names)}

# Converting userful info to named tuples
## Creating a seperate config_nt_class and train_test_nt_class as they are reused in the tuning loop
config_nt_class = dictionary_to_named_tuple_class('config_nt', config_dict)
config_nt = config_nt_class(**config_dict)
train_test_nt_class = dictionary_to_named_tuple_class('train_test_nt', train_test_dict)
train_test_nt = train_test_nt_class(**train_test_dict)

# For other dicts we dont need to save the class
output_idx_nt = dictionary_to_named_tuple_class('output_idx_nt', output_idx_dict)(**output_idx_dict)
model_idx_nt = dictionary_to_named_tuple_class('model_idx_nt', model_idx_dict)(**model_idx_dict)
bin_metric_nt = dictionary_to_named_tuple_class('bin_metric_nt', bin_metric_dict)(**bin_metric_dict)

### Creating helper functions for the final runner

#### Math helper functions

In [ ]:
from numba import njit
import math 

### Helper functions to stop overflow
@njit(inline='always')
def logsumexp2(a, b):
    ''' 
    Two term logsumexp helper
    '''
    if a == b:
        return a + math.log(2.0)
    if a > b:
        return a + math.log1p(math.exp(b - a))
    else:
        return b + math.log1p(math.exp(a - b))


log_0_5 = -math.log(2.0)

@njit(inline="always")
def log1minexp(log_p):
    """
    Stable log(1 - exp(log_p))
    """

    if log_p < log_0_5:
        return math.log1p(-math.exp(log_p))
    else:
        return math.log(-math.expm1(log_p))

In [ ]:
# Creating functions that get the log pmf value for the negative binomial distribution (or the poisson distirbution for underdispersed users)
### LOG PMF VALUES
@njit 
def poisson_lpmf(x, mu):
    ''' 
    Poisson log pmf
    '''
    return x*math.log(mu) - mu - math.lgamma(x+1)

@njit
def neg_bin_lpmf(x, mu, sigma2):
    p = mu/sigma2
    r = (mu*p) / (1-p)
    return math.lgamma(x+r) - math.lgamma(r) - math.lgamma(x+1) + r*math.log(p) + x*math.log(1-p)

@njit 
def hurdle_lpmf(x, mu, sigma2, p, config_nt):
    '''
    Note the poisson fallback is handled as we use the nb functions with poisson integrated
    '''
    if x == 0:
        return math.log1p(-p)
    else:
        return math.log(p) + get_nb_lpmf_val(x, mu, sigma2, config_nt) - log1minexp(get_nb_lpmf_val(0, mu, sigma2, config_nt))

@njit 
def get_nb_lpmf_val(x, mu, sigma2, config_nt):
    if mu <= 0:
        raise ValueError('Mu < 0')
    if sigma2 <= 0:
        raise ValueError('Sigma^2 < 0')
    
    mu = max(mu, config_nt.mean_min)
    sigma2 = max(sigma2, config_nt.var_min)
    if sigma2 <= mu + config_nt.min_mean_var_diff:
        return poisson_lpmf(x, mu)
    else: 
        return neg_bin_lpmf(x, mu, sigma2)

@njit(inline='always')
def get_lpmf_val(x, mu, sigma2, p, config_nt):
    ''' 
    Gets the LPMF value for the count
    '''
    if config_nt.hurdle_model:
        return hurdle_lpmf(x, mu, sigma2, p, config_nt)
    else:
        return get_nb_lpmf_val(x, mu, sigma2, config_nt)

### UPPER TAIL VALUES
@njit 
def poisson_log_upper_tail(x, mu):
    ''' 
    Gets p(X>= x) for a poisson distribution 
    '''
    if x == 0:
        return 0
    
    # Looping over k and getting prob x = k and adding to lower tail
    log_prob_k = -mu
    lower_tail = log_prob_k
    for k in range(1,x):
        log_prob_k = log_prob_k + math.log(mu) - math.log(k)
        lower_tail = logsumexp2(lower_tail, log_prob_k)

    return log1minexp(lower_tail)


@njit 
def neg_bin_log_upper_tail(x, mu, sigma2):
    ''' 
    Gets p(X>= x) for a negative binomial distribution 
    '''
    if x == 0:
        return 0
    
    p = mu/sigma2
    r = (mu*p) / (1-p)

    # Looping over k and getting prob x = k and adding to lower tail
    log_prob_k = r * math.log(p)
    log_lower_tail = log_prob_k
    for k in range(1,x):
        log_prob_k = log_prob_k+ math.log((k-1)+r) - math.log(k) + math.log(1-p)
        log_lower_tail =  logsumexp2(log_lower_tail, log_prob_k)

    return log1minexp(log_lower_tail)

@njit 
def hurdle_upper_tail(x, mu, sigma2, p, config_nt):
    if x == 0:
        return 0
    else:
        return math.log(p) + get_nb_upper_tail_value(x, mu, sigma2, config_nt) - log1minexp(get_nb_lpmf_val(0, mu, sigma2, config_nt))
    
@njit 
def get_nb_upper_tail_value(x, mu, sigma2, config_nt):
    ''' 
    We dont need edge case checks here as the other function is called with the same mu and sigma
    '''
    mu = max(mu, config_nt.mean_min)
    sigma2 = max(sigma2, config_nt.var_min)
    if sigma2 <= mu + config_nt.min_mean_var_diff:
        return poisson_log_upper_tail(x, mu)
    else: 
        return neg_bin_log_upper_tail(x, mu, sigma2)
    
    
@njit(inline='always')
def get_upper_tail_value(x, mu, sigma2, p, config_nt):
    if config_nt.hurdle_model:
        return hurdle_upper_tail(x, mu, sigma2, p, config_nt)
    else:
        return get_nb_upper_tail_value(x, mu, sigma2, config_nt)

#### Helper function for metrics

In [ ]:
@njit
def update_outputs(time_period_int, output_metrics, calibration_output, output_idx_nt, model_idx_nt,
                  log_calibration_thresholds, log_upper_tail_raw, log_upper_tail_smoothed, lpmf_raw, lpmf_smoothed):
    ''' 
    Function used for updating the output and calibration threshold output
    Args:
        x : the count observed
        time_period_int : number that states whether we are in test train or validation
        output_metrics : Np array where we will store our outputs
        calibration_output : output of how many p values fall below each threshold in train and validation for each model
        output_idx_nt : named tuple that tells us which index of `output_metrics` each metric lives in
        model_idx_nt :  named tuple that tells us which index of `calibration_output` each model lives in

        log_raw_degen_threshold : threshold that tells us whether we are in a degenerate bin or not (defined by P(X=0))
        log_calibration_thresholds : calibration thresholds we are monitoring (how many p values fall below each threshold)

        log_p0_raw  + smoothed: the log prob of 0 is compared to the degen threshold in the function
        log_upper_tail_raw + smooth : the log_probability of observing a value greater that or equal to x. compared to the log calibration thresholds

        lpmf_raw + smoothed : the log likelihood values we observe
    '''

    # If we are in train or validation update the metrics
    if time_period_int == 0 or time_period_int == 1:
        output_metrics[time_period_int, output_idx_nt.n_bins_scored] += 1
        output_metrics[time_period_int, output_idx_nt.non_degen_ll_sum] += lpmf_raw
        output_metrics[time_period_int, output_idx_nt.non_degen_smoothed_ll_sum] += lpmf_smoothed

        # Add a point for each calibration threshold we are less than 
        # 1 row of output for raw model one for smoothed model
        for calib_threshold_idx in range(log_calibration_thresholds.shape[0]):

            if log_upper_tail_raw < log_calibration_thresholds[calib_threshold_idx]:
                calibration_output[time_period_int, calib_threshold_idx, model_idx_nt.raw_model_calib_index] += 1

            if log_upper_tail_smoothed < log_calibration_thresholds[calib_threshold_idx]:
                calibration_output[time_period_int, calib_threshold_idx, model_idx_nt.smoothed_model_calib_index] += 1

#### Time period function

In [23]:
@njit
def get_time_period(fine_bin_id, validation_start, validation_end, test_start, test_end):
    ''' 
    Returns:
        0 if we are in the validation data
        1 if we are in the test data
        -1 otherwise (train + burn in and any unused data)
    '''
    if validation_start <= fine_bin_id and fine_bin_id < validation_end:
        return 0
    elif test_start <= fine_bin_id and fine_bin_id < test_end:
        return 1
    else: 
        return -1

#### Creating and updating alpha grids

In [ ]:
## Creating a grid of alpha for smoothign

def init_alpha_grid(n_counts_init, config_dict):
    ''' 
    Init a grid of alpha values that are used in smoothing
    '''
    # Return a constant grid if we are using linear smoothing else make a count dependent alpha
    if config_dict['linear_smooth']:
        return np.full_like(n_counts_init, config_dict['smooth_a'], dtype='float64')
    else: 
        return config_dict['smooth_k'] / (np.log1p(n_counts_init) + config_dict['smooth_k'])
    
@njit
def update_alpha_grid(n_counts, config_nt):
    ''' 
    Gets a value of alpha fron n_counts similarly to `init_alpha_grid`
    '''
    if config_nt.linear_smooth:
        return config_nt.smooth_a
    else: 
        return config_nt.smooth_k / (math.log1p(n_counts) + config_nt.smooth_k)

@njit
def update_n_counts_and_alpha_grid(n_counts, alpha_grid, crnt_user_id, usr_updt_n_counts, config_nt):
    ''' 
    Runs at the end of every week in the ll runner
    '''
    n_coarse_bins = n_counts.shape[1]

    # Iterating over the grid adding the weekly sums and updating the grid
    for coarse_bin in range(n_coarse_bins):
        n_counts[crnt_user_id, coarse_bin] += usr_updt_n_counts[coarse_bin]

        alpha_grid[crnt_user_id, coarse_bin] = update_alpha_grid(n_counts[crnt_user_id, coarse_bin], config_nt)
    
alpha_grid_init = init_alpha_grid(n_counts_init, config_dict)

#### Smoothing helpers

In [ ]:
# Creating a function that smooths between users and parameter
# TODO maybe remove this function later
# @njit(inline='always')
# def get_smoothing_alpha(config_nt, n_user_samples):
#     ''' 
#     Gets the value of alpha used for smoothign
#     '''
#     if config_nt.linear_smooth:
#         return config_nt.smooth_a

#     return config_nt.smooth_k / (math.log1p(n_user_samples) + config_nt.smooth_k)

@njit(inline='always')
def smoothing_function(alpha, user_parameter, target_parameter):
    return (1.0 - alpha) * user_parameter + alpha * target_parameter


@njit 
def interpolate_values(v_neg_1, v_0, v_1, fine_bin_within_coarse_pos, interpolation_weights):
    '''
        Applies the quadratic interpolation between the left middle and right bin values
    '''
    # Extract the weights we will use for the interpolation
    w_neg_1, w_0, w_1 = interpolation_weights[fine_bin_within_coarse_pos]
    return w_neg_1 * v_neg_1 + w_0 * v_0 + w_1 * v_1

@njit 
def smooth_params(user_param_grid, cluster_param_grid, cluster_assignments, user_totals, cluster_totals, 
                  alpha_grid, crnt_user, crnt_coarse_bin, crnt_fine_bin_within_coarse_pos, interpolation_weights, config_nt):
    ''' 
    Takes a parameter mu or sigma2 and smooths it towards the cluster parameter using `smoothing_function`
    Then interpolates with interpolate values
    '''

    # Get the 3 values to interpolate
    n_coarse_bins = user_param_grid.shape[1]
    neg_1_coarse_bin = (crnt_coarse_bin -1) % n_coarse_bins
    _1_coarse_bin = (crnt_coarse_bin + 1)% n_coarse_bins

    # Getting the values of alpha for smoothing
    alpha_neg_1 = alpha_grid[crnt_user, neg_1_coarse_bin]
    alpha_0 = alpha_grid[crnt_user, crnt_coarse_bin]
    alpha_1 = alpha_grid[crnt_user, _1_coarse_bin]

    # Getting the cluster targets (values we smooth towards)
    target_neg_1 = get_cluster_target(user_param_grid, cluster_param_grid, cluster_assignments, user_totals, cluster_totals, crnt_user, neg_1_coarse_bin, config_nt)
    target_0 = get_cluster_target(user_param_grid, cluster_param_grid, cluster_assignments, user_totals, cluster_totals, crnt_user, crnt_coarse_bin, config_nt)
    target_1 = get_cluster_target(user_param_grid, cluster_param_grid, cluster_assignments, user_totals, cluster_totals, crnt_user, _1_coarse_bin, config_nt)

    ## Smooth the 3 values we will later interpolate if alpha is non zero
    if alpha_neg_1 == 0:
        v_neg_1 = user_param_grid[crnt_user, neg_1_coarse_bin]
    else:
        v_neg_1 = smoothing_function(alpha_neg_1, user_param_grid[crnt_user, neg_1_coarse_bin], target_neg_1)
    if alpha_0 == 0:
        v_0 = user_param_grid[crnt_user, crnt_coarse_bin]
    else:
        v_0 = smoothing_function(alpha_0, user_param_grid[crnt_user, crnt_coarse_bin], target_0)
    if alpha_1 == 0:
        v_1 =  user_param_grid[crnt_user, _1_coarse_bin]
    else:
        v_1 = smoothing_function(alpha_1, user_param_grid[crnt_user, _1_coarse_bin], target_1)

    # Interpolate the values 
    return interpolate_values(v_neg_1, v_0, v_1, crnt_fine_bin_within_coarse_pos, interpolation_weights)

@njit 
def get_cluster_target(user_grid, cluster_grid, cluster_assignments, user_totals, cluster_totals, crnt_user, crnt_coarse_bin, config_nt):
    ''' 
    Returns a variable to pass to smooth values for the cluster value
    '''
    # Get the assignment val for user and cluster
    cluster_id = cluster_assignments[crnt_user]
    user_val = user_grid[crnt_user, crnt_coarse_bin]
    cluster_val = cluster_grid[cluster_id, crnt_coarse_bin]

    user_total = user_totals[crnt_user]
    cluster_total =cluster_totals[cluster_id] 

    # Absolute smoothing (we smooth towards tehc cluster value)
    if config_nt.smoothing_target == 0:
        return cluster_val

    # Shape smoothing (we smooth towards the custer shape while keeping the user rate)
    if config_nt.smoothing_target == 1:
        cluster_shape = cluster_val / cluster_total
        return user_total * cluster_shape

    # Rate smoothing we smooth towards cluster rate keeping the users shape
    if config_nt.smoothing_target == 2:
        user_shape = user_val / user_total
        return cluster_total * user_shape

@njit
def get_smoothed_params(u, v, p, cluster_u, cluster_v, cluster_p, cluster_assignments, 
                        user_u_totals, user_v_totals, user_p_totals, cluster_u_totals, cluster_v_totals, cluster_p_totals, 
                        alpha_grid, crnt_user, crnt_coarse_bin, crnt_fine_bin_within_coarse_pos, interpolation_weights, config_nt):
    '''
    Smooths my sigma2 and p if applicable
    '''
    
    mu = smooth_params(u, cluster_u, cluster_assignments, user_u_totals, cluster_u_totals, 
                       alpha_grid, crnt_user, crnt_coarse_bin, crnt_fine_bin_within_coarse_pos, interpolation_weights, config_nt)

    sigma2 = smooth_params(v, cluster_v, cluster_assignments, user_v_totals, cluster_v_totals, 
                           alpha_grid, crnt_user, crnt_coarse_bin, crnt_fine_bin_within_coarse_pos, interpolation_weights, config_nt)

    if config_nt.hurdle_model:
        p_val = smooth_params(p, cluster_p, cluster_assignments, user_p_totals, cluster_p_totals, 
                              alpha_grid, crnt_user, crnt_coarse_bin, crnt_fine_bin_within_coarse_pos, interpolation_weights, config_nt)
    else:
        p_val = 0.0

    return mu, sigma2, p_val

#### Creating new grid and new cluster means

In [ ]:
# Creating a function that collects grid updates and a function that updates the grid

@njit
def update_grid(u, v, p, crnt_user_id, usr_updt_u_sum, usr_updt_v_sum, usr_updt_p_sum, usr_updt_pos_sum, fine_bins_per_coarse_bin, config_nt):
    ''' 
    Replaces the grid values using the temporary grid as data comes in
    '''
    # Update hurdle model grid
    if config_nt.hurdle_model:
        n_coarse_bins = u.shape[1]

        for coarse_bin in range(n_coarse_bins):
            p_new = usr_updt_p_sum[coarse_bin] / fine_bins_per_coarse_bin
            p[crnt_user_id, coarse_bin] = min(max(p_new, config_nt.p_min), config_nt.p_max)

            # Only update grid if we have a positive count in that coaraes bin
            if usr_updt_pos_sum[coarse_bin] > 0:
                u[crnt_user_id, coarse_bin] = max(usr_updt_u_sum[coarse_bin] / usr_updt_pos_sum[coarse_bin], config_nt.mean_min)
                v[crnt_user_id, coarse_bin] = max(usr_updt_v_sum[coarse_bin] / usr_updt_pos_sum[coarse_bin], config_nt.var_min)

    # Update the grid for the NB model 
    else:
        u[crnt_user_id, : ] = usr_updt_u_sum/fine_bins_per_coarse_bin
        v[crnt_user_id, : ] = usr_updt_v_sum/fine_bins_per_coarse_bin

@njit
def collect_temp_grid(usr_updt_u_sum, usr_updt_v_sum, usr_updt_p_sum, usr_updt_pos_sum, usr_updt_cnt_sum, crnt_coarse_bin, x, mu_t, sigma_2_t, p_t, config_nt):
    '''
    As data comes in we update the interpolated mu values by combining with incoming data as per lambert and liu formula
    returns nothing as we modify in place. Also has an update version for hurdle model using ll formula + new formula for p
    '''
    # updating the count sum
    usr_updt_cnt_sum[crnt_coarse_bin] += x
    
    # Update logic for the hurdle model
    if config_nt.hurdle_model:
        # Init a binary variable that detects x >0 
        if x > 0:
            z= 1
        else:
            z = 0

        # Updating p value and sum of p vals in the bin   
        p_new = (1 - config_nt.w) * p_t + config_nt.w * z
        p_new = min(max(p_new, config_nt.p_min), config_nt.p_max)
        usr_updt_p_sum[crnt_coarse_bin] += p_new

        # Update other values if x > 0
        if x > 0:
            mu_new = (1-config_nt.w)*mu_t + config_nt.w*x
            sigma2_new = (1-config_nt.w)*sigma_2_t + config_nt.w*(x-mu_t)*(x-mu_new)

            mu_new = max(mu_new, config_nt.mean_min)
            sigma2_new = max(sigma2_new, config_nt.var_min)

            usr_updt_u_sum[crnt_coarse_bin] += mu_new
            usr_updt_v_sum[crnt_coarse_bin] += sigma2_new
            usr_updt_pos_sum[crnt_coarse_bin] += 1

    # Update logic for the NB model
    else: 
        mu_new = (1-config_nt.w)*mu_t + config_nt.w*x
        sigma2_new = (1-config_nt.w)*sigma_2_t + config_nt.w*(x-mu_t)*(x-mu_new)

        mu_new = max(mu_new, config_nt.mean_min)
        sigma2_new = max(sigma2_new, config_nt.var_min)

        usr_updt_u_sum[crnt_coarse_bin] += mu_new
        usr_updt_v_sum[crnt_coarse_bin] += sigma2_new
    

In [ ]:
# Function for updating cluster parameters 
@njit 
def get_new_clustering_means(cluster_groups, u, v, p):
    ''' 
    Calculates mean u and v values for each cluster group and each time bin
    Args:
        cluster_groups a n_users length vector of cluster assignments
        u : the calculated vector of u parameter means
        v : the calculated vector of v parameter variances
    '''

    n_users, n_coarse_bins = u.shape 
    n_clusters = cluster_groups.max() + 1

    # Init mean vectors
    cluster_mean_u = np.zeros((n_clusters, n_coarse_bins), dtype=np.float64)
    cluster_mean_v = np.zeros((n_clusters, n_coarse_bins), dtype=np.float64)
    cluster_mean_p = np.zeros((n_clusters, n_coarse_bins), dtype=np.float64)

    users_per_cluster = np.zeros(n_clusters, dtype=np.float64)

    # Summing u and v contributions in each cluster
    # Extract the cluster assignment for each user and then add their parameters to each bin
    for user_id in range(n_users):
        cluster_assignment = int(cluster_groups[user_id])

        cluster_mean_u[cluster_assignment, :] += u[user_id, :]
        cluster_mean_v[cluster_assignment, :] += v[user_id, :]
        cluster_mean_p[cluster_assignment, :] += p[user_id, :]

        users_per_cluster[cluster_assignment] += 1

    # Dividing through to get the averages in each cluster
    for cluster_assignment in range(n_clusters):
        if users_per_cluster[cluster_assignment] > 0:
            cluster_mean_u[cluster_assignment, :] /= users_per_cluster[cluster_assignment]
            cluster_mean_v[cluster_assignment, :] /= users_per_cluster[cluster_assignment]
            cluster_mean_p[cluster_assignment, :] /= users_per_cluster[cluster_assignment]

    return cluster_mean_u, cluster_mean_v, cluster_mean_p

### Making LL more modular

In [ ]:
@njit(inline='always')
def _init_user_count_table_pointer(n_users, user_interactions_nt, user_counts_nt, train_test_nt):
    ''' 
    Initialises a pointer which points to the first non train row in user_counts for each user.
    '''
    usr_frst_rw = user_interactions_nt.user_first_index.copy()

    for user_id in range(n_users):
        cnt_tbl_idx = user_interactions_nt.user_first_index[user_id]
        usr_lst_idx = user_interactions_nt.user_last_index[user_id]
        
        while cnt_tbl_idx <= usr_lst_idx and user_counts_nt.fine_bin_id[cnt_tbl_idx] < train_test_nt.burn_in_start:
            cnt_tbl_idx +=1
        usr_frst_rw[user_id] = cnt_tbl_idx
    return usr_frst_rw

@njit(inline='always')
def _get_user_count(cnt_tbl_idx, user_counts_nt, usr_end_idx, fine_bin_idx):
    ''' 
    Gets the count for that bin and moves the pointer if needed
    '''
    if cnt_tbl_idx <= usr_end_idx and user_counts_nt.fine_bin_id[cnt_tbl_idx] == fine_bin_idx:
        x = user_counts_nt.count[cnt_tbl_idx]
        cnt_tbl_idx += 1
    else:
        x = 0
    return x, cnt_tbl_idx

@njit(inline='always')
def _get_smoothed_and_unsmoothed_params(u, v, p, cluster_u, cluster_v, cluster_p, cluster_groups, 
                                        user_u_totals, user_v_totals, user_p_totals, cluster_u_totals, cluster_v_totals, cluster_p_totals, 
                                        alpha_grid, alpha_zero_grid, user_id, crnt_coarse_bin, crnt_fine_bin_within_coarse_pos, interpolation_weights, config_nt):
    ''' 
    Getting the values of mu and sigma for both the raw and smoothed model
    '''
    # Getting the smoothed params and capping them at the minimal value
    mu_t, sigma_2_t, p_t = get_smoothed_params(u, v, p, cluster_u, cluster_v, cluster_p, cluster_groups, 
                                            user_u_totals, user_v_totals, user_p_totals, cluster_u_totals, cluster_v_totals, cluster_p_totals, 
                                            alpha_grid, user_id, crnt_coarse_bin, crnt_fine_bin_within_coarse_pos, interpolation_weights, config_nt)
    

    # Getting unsmoothed but interpolated params and using that for updates (difference from above call is passing smoothing strength 0):
    mu_unsmth_t, sigma_unsmth_2_t, p_unsmth_t = get_smoothed_params(u, v, p, cluster_u, cluster_v, cluster_p, cluster_groups, 
                                            user_u_totals, user_v_totals, user_p_totals, cluster_u_totals, cluster_v_totals, cluster_p_totals, 
                                            alpha_zero_grid, user_id, crnt_coarse_bin, crnt_fine_bin_within_coarse_pos, interpolation_weights, config_nt)
    
    # Capping values
    mu_t = max(mu_t, config_nt.mean_min)
    sigma_2_t = max(sigma_2_t, config_nt.var_min)
    mu_unsmth_t = max(mu_unsmth_t, config_nt.mean_min)
    sigma_unsmth_2_t = max(sigma_unsmth_2_t, config_nt.var_min)
    p_t = min(max(p_t, config_nt.p_min), config_nt.p_max)
    p_unsmth_t = min(max(p_unsmth_t, config_nt.p_min), config_nt.p_max)

    return mu_t, sigma_2_t, p_t, mu_unsmth_t, sigma_unsmth_2_t, p_unsmth_t

@njit(inline='always')
def _bin_computations(bin_metric_nt, fine_bin_idx):
    ''' 
    Computes coarse bin and fine bin within the week and coarse bin
    '''
    fine_bin_pos_in_week = fine_bin_idx % bin_metric_nt.fine_bins_per_week
    crnt_coarse_bin = fine_bin_pos_in_week // bin_metric_nt.fine_bins_per_coarse_bin
    crnt_fine_bin_within_coarse_pos = fine_bin_pos_in_week % bin_metric_nt.fine_bins_per_coarse_bin
    return crnt_coarse_bin, crnt_fine_bin_within_coarse_pos

@njit(inline='always')
def _get_log_p0_lpmf_and_upper_tail(x, mu_t, sigma_2_t, p_t, mu_unsmth_t, sigma_unsmth_2_t, p_unsmth_t, config_nt):
    ''' 
    Returns log p0, the lpmf, and upper tail values for the smoothed and raw models
    '''
    lpmf_smoothed = get_lpmf_val(x, mu_t, sigma_2_t, p_t, config_nt)
    lpmf_raw = get_lpmf_val(x, mu_unsmth_t, sigma_unsmth_2_t, p_unsmth_t, config_nt)

    # Not currently used but might be used in future diagnostics.
    # if x == 0:
    #     log_p0_raw = lpmf_raw
    #     log_p0_smoothed = lpmf_smoothed
    # else: 
    #     log_p0_raw = get_lpmf_val(0, mu_unsmth_t, sigma_unsmth_2_t, p_unsmth_t, config_nt)
    #     log_p0_smoothed = get_lpmf_val(0, mu_t, sigma_2_t, p_t, config_nt)

    # Getting the upper tail value for both the raw and the smoothed model
    log_upper_tail_raw = get_upper_tail_value(x, mu_unsmth_t, sigma_unsmth_2_t, p_unsmth_t, config_nt)
    log_upper_tail_smoothed = get_upper_tail_value(x, mu_t, sigma_2_t, p_t, config_nt)
    
    return lpmf_raw, lpmf_smoothed, log_upper_tail_raw, log_upper_tail_smoothed # ,log_p0_raw, log_p0_smoothed

#### More LL functions

In [ ]:
@njit
def get_grid_row_sums(grid):
    ''' 
    Gets the sum of a parameter grid
    Needed for seperating the shape from the rate and vice versa
    '''
    # Extract the number of rows and init the ouptu gird
    n_rows, n_cols = grid.shape
    output = np.zeros(n_rows, dtype='float64')

    # Loop over and count element contributions
    for row_idx in range(n_rows):
        total = 0
        for col_idx in range(n_cols):
            total += grid[row_idx, col_idx]
        output[row_idx] = total
    return output

#### Lambert liu runner

In [ ]:
@njit
def run_lambert_liu(u_init, v_init, p_init, cluster_u_init, cluster_v_init, cluster_p_init, cluster_groups, n_counts_init, 
                    alpha_grid_init, degen_mask, user_counts_nt, user_interactions_nt, interpolation_weights,
                    train_test_nt, bin_metric_nt, config_nt,
                    output_idx_nt, model_idx_nt):
    ''' 
    Runs the lambert liu algorithm
    Args:
        u_init + v_init : inital parameter grids
        cluster_u_init + cluster_v_init : inital cluster parameters
        cluster groups : inital cluster assignments 1 row per user id
        smooth_a : parameter (experiment to vary)
        user_counts_nt: is a named tuple version of user_counts has columns user_id, fine_bin_id, count
        user_interactions_nt: is a named tuple verion of user_interactions has columns user id and first and last interaction index in user_counts
        interpolation_weights : precalculated weights for parameter interpolation
        train_test_nt, bin_metric_nt, config_dt : named tuple versions of dicts train_test_dict, config_dict and bin_metric_dict
        output_idx_nt : a named tuple containing the names and indicies of the outputs we want to store
        model_idx_nt : a named tuple containing the names and indicies where we store each model outputs
    '''
    ###
    # Initialising grids where we will keep track of parameters and cluster mean parameters
    u = u_init.copy()
    v = v_init.copy()
    p = p_init.copy()

    cluster_u = cluster_u_init.copy()
    cluster_v = cluster_v_init.copy()
    cluster_p = cluster_p_init.copy()

    n_counts = n_counts_init.copy()
    alpha_grid = alpha_grid_init.copy()
    zero_alpha_grid = np.zeros_like(alpha_grid)


    # Calculating needed values
    n_users, n_coarse_bins = u.shape

    log_calibration_thresholds = np.log(config_nt.calibration_thresholds)

    burn_in_first_week = train_test_nt.burn_in_start // bin_metric_nt.fine_bins_per_week
    test_last_week = (train_test_nt.test_end-1)// bin_metric_nt.fine_bins_per_week

    # Init outputs
    output_metrics = np.zeros((2, len(output_idx_nt)), dtype='float64')
    calibration_output = np.zeros((2, log_calibration_thresholds.shape[0], len(model_idx_nt)), dtype='float64')

    # Init pointer for user interactions
    usr_frst_rw = _init_user_count_table_pointer(n_users, user_interactions_nt, user_counts_nt, train_test_nt)

    for week in range(burn_in_first_week, test_last_week + 1):
        
        week_start = week * bin_metric_nt.fine_bins_per_week
        week_end = (week + 1) * bin_metric_nt.fine_bins_per_week

        if week_end > train_test_nt.test_end:
            week_end = train_test_nt.test_end

        # Getting grid totals for seperating rate from shape
        user_u_totals = get_grid_row_sums(u)
        user_v_totals = get_grid_row_sums(v)
        user_p_totals = get_grid_row_sums(p)

        cluster_u_totals = get_grid_row_sums(cluster_u)
        cluster_v_totals = get_grid_row_sums(cluster_v)
        cluster_p_totals = get_grid_row_sums(cluster_p)

        # For each week iterate over the users and init the pointers
        for user_id in range(n_users):

            cnt_tbl_idx = usr_frst_rw[user_id]
            usr_end_idx = user_interactions_nt.user_last_index[user_id]

            # Init numpy vectors for calculating the user sums
            usr_updt_u_sum = np.zeros(n_coarse_bins, dtype=np.float64)
            usr_updt_v_sum = np.zeros(n_coarse_bins, dtype=np.float64)
            usr_updt_p_sum = np.zeros(n_coarse_bins, dtype=np.float64)
            
            usr_updt_pos_sum = np.zeros(n_coarse_bins, dtype=np.float64)
            usr_updt_cnt_sum = np.zeros(n_coarse_bins, dtype=np.float64)

            for fine_bin in range(week_start, week_end):
                
                x, cnt_tbl_idx = _get_user_count(cnt_tbl_idx, user_counts_nt, usr_end_idx, fine_bin)
                
                crnt_coarse_bin, crnt_fine_bin_within_coarse_pos = _bin_computations(bin_metric_nt, fine_bin)

                mu_t, sigma_2_t, p_t, mu_unsmth_t, sigma_unsmth_2_t, p_unsmth_t = _get_smoothed_and_unsmoothed_params(u, v, p, cluster_u, cluster_v, cluster_p, 
                                                                                    cluster_groups, user_u_totals, user_v_totals, user_p_totals, cluster_u_totals, cluster_v_totals, cluster_p_totals, 
                                                                                    alpha_grid, zero_alpha_grid, user_id, crnt_coarse_bin, crnt_fine_bin_within_coarse_pos, interpolation_weights, config_nt)

                time_period_int = get_time_period(fine_bin, train_test_nt.validation_start, train_test_nt.validation_end, 
                                                                train_test_nt.test_start, train_test_nt.test_end)
                
                if (time_period_int == 0 or time_period_int == 1) and not degen_mask[user_id, crnt_coarse_bin]: 
                    lpmf_raw, lpmf_smoothed, log_upper_tail_raw, log_upper_tail_smoothed = _get_log_p0_lpmf_and_upper_tail(x, mu_t, sigma_2_t, p_t, 
                                                                                                                            mu_unsmth_t, sigma_unsmth_2_t, p_unsmth_t, config_nt)
                    # Updating outputs
                    update_outputs(time_period_int, output_metrics, calibration_output, output_idx_nt, model_idx_nt, log_calibration_thresholds, log_upper_tail_raw, log_upper_tail_smoothed, lpmf_raw, lpmf_smoothed)

                collect_temp_grid(usr_updt_u_sum, usr_updt_v_sum, usr_updt_p_sum, usr_updt_pos_sum, usr_updt_cnt_sum, crnt_coarse_bin, x, mu_unsmth_t, sigma_unsmth_2_t, p_unsmth_t, config_nt)
                            
            # Updating the users first row (for the next week) and the parameter grid and alpha grid
            usr_frst_rw[user_id] = cnt_tbl_idx
            update_grid(u, v, p, user_id, usr_updt_u_sum, usr_updt_v_sum, usr_updt_p_sum, usr_updt_pos_sum, bin_metric_nt.fine_bins_per_coarse_bin, config_nt)
            update_n_counts_and_alpha_grid(n_counts, alpha_grid, user_id, usr_updt_cnt_sum, config_nt)

        cluster_u, cluster_v, cluster_p = get_new_clustering_means(cluster_groups, u, v, p)

    return output_metrics, calibration_output, u, v, p, cluster_u, cluster_v, cluster_p, n_counts, alpha_grid

In [ ]:
def run_pipeline_ll(model, config_nt, train_test_nt, config_dict, degen_mask):
    ''' 
    Makes a call to the numba lambert liu runner
    '''
    u, v, p, _, _, _ = get_ll_param_grids(config_dict)

    return run_lambert_liu(
        u_init=u,
        v_init=v,
        p_init=p,
        cluster_u_init=model['cluster_mean_u'],
        cluster_v_init=model['cluster_mean_v'],
        cluster_p_init=model['cluster_mean_p'],
        cluster_groups=model['cluster_assignments'],
        n_counts_init=n_counts_init,
        alpha_grid_init=init_alpha_grid(n_counts_init, config_dict),
        degen_mask=degen_mask,
        user_counts_nt=user_counts_nt,
        user_interactions_nt=user_interactions_nt,
        interpolation_weights=interpolation_weights,
        train_test_nt=train_test_nt,
        bin_metric_nt=bin_metric_nt,
        config_nt=config_nt,
        output_idx_nt=output_idx_nt,
        model_idx_nt=model_idx_nt)


In [ ]:
output_metrics, calibration_outputs, *_ = run_pipeline_ll(clustering_model, config_nt=config_nt, train_test_nt=train_test_nt, config_dict=config_dict, degen_mask=degen_mask)

### Output Table Creation

In [ ]:
def make_output_table_row(model, output_metrics, config_dict, test_valid):
    ''' 
    Transforms the output row into a readable dictionary that can be used as an output df row.
    Args:
        test_valid : Can take values `test` and `valid` tells us what period we are in

    '''

    # Init variables
    if test_valid == 'valid':
        period_idx = 0
    elif test_valid == 'test':
        period_idx = 1
    else:
        raise ValueError('test_valid must either be `test` or `valid`')
    n_non_degen_bins = output_metrics[period_idx, output_idx_nt.n_bins_scored]
    

    output = {
        # Row descriptions
        'smoothed_model_name': model['name'],
        'w': config_dict['w'],
        'cluster_param': model['cluster_param'],
        'smooth_a': config_dict['smooth_a'],
        'smooth_k': config_dict['smooth_k'],
        'linear_smooth': config_dict['linear_smooth'],
        'smoothing_target': config_dict['smoothing_target'],
        'hurdle_model': config_dict['hurdle_model'],
        'test_valid' : test_valid,

        # Log likelihood for non degenerate bins
        'non_degen_ll': output_metrics[period_idx, output_idx_nt.non_degen_ll_sum] / n_non_degen_bins,
        'non_degen_smoothed_ll': output_metrics[period_idx, output_idx_nt.non_degen_smoothed_ll_sum] / n_non_degen_bins,
        
        # Clustering metrics
        'clustering_matrix_name': model['clustering_matrix_name'],
        'seed': model['seed'],
        'cluster_inertia': model['cluster_inertia'],}

    return output



def make_calibration_output_rows(model, output_metrics, calibration_outputs, test_valid, config_dict):
    ''' 
    Creates a table of calibration outputs
    '''

    # Init variables and outputs
    if test_valid == 'valid':
        period_idx = 0
    elif test_valid == 'test':
        period_idx = 1
    else:
        raise ValueError('test_valid must either be `test` or `valid`')
    
    output = []
    n_non_degen_bins = output_metrics[period_idx, output_idx_nt.n_bins_scored]

    for threshold_idx in range(config_dict['calibration_thresholds'].shape[0]):

        ## Appending a row to output
        output.append({
            # Row descriptions
            'smoothed_model_name': model['name'],
            'w': config_dict['w'],
            'cluster_param': model['cluster_param'],
            'smooth_a': config_dict['smooth_a'],
            'smooth_k': config_dict['smooth_k'],
            'linear_smooth': config_dict['linear_smooth'],
            'smoothing_target': config_dict['smoothing_target'],
            'hurdle_model': config_dict['hurdle_model'],
            'clustering_matrix_name': model['clustering_matrix_name'],
            'test_valid': test_valid,

            # Calibration metrics
            'threshold': config_dict['calibration_thresholds'][threshold_idx],
            'raw_tail_rate': calibration_outputs[period_idx, threshold_idx, model_idx_nt.raw_model_calib_index] / n_non_degen_bins,
            'smoothed_tail_rate': calibration_outputs[period_idx, threshold_idx, model_idx_nt.smoothed_model_calib_index] / n_non_degen_bins,
        })

    return output

### Creating a validation hyperparameter tuning loop

In [ ]:
# function that adds hyperparamter configurations to config dict for run

def make_temp_config(config_dict, w, cluster_param, hurdle_model, smoothing_target, linear_smooth, smooth_a, smooth_k):
    ''' 
    Adds a set of hyperparameters to a temporary copy of the config dict
    '''
    temp_config = config_dict.copy()

    temp_config['w'] = w
    temp_config['cluster_param'] = cluster_param
    temp_config['hurdle_model'] = hurdle_model
    temp_config['smoothing_target'] = smoothing_target
    temp_config['linear_smooth'] = linear_smooth
    temp_config['smooth_a'] = smooth_a
    temp_config['smooth_k'] = smooth_k

    return temp_config

In [ ]:
def create_tuning_dict(dict):
    ''' 
    Returns a dictionary without a test period used for hyperparameter tuning
    '''
    validation_only_dict = dict.copy()
    validation_only_dict['test_start'] = validation_only_dict['validation_end']
    validation_only_dict['test_end'] = validation_only_dict['validation_end']
    return validation_only_dict

def tune_models(hyperparams, train_test_dict, config_dict, degen_mask, config_nt_class=config_nt_class):
    ''' 
    Runs the hyperparameter tuning for the cluster based smoothing model and the raw model
    '''
    validation_only_dict = create_tuning_dict(train_test_dict)
    validation_only_nt = train_test_nt_class(**validation_only_dict)

    results = []
    for w in hyperparams['w_vals']:
        for cluster_param in hyperparams['cluster_param_vals']:
            for hurdle_model in hyperparams['hurdle_model']:
                for smoothing_target in hyperparams['smoothing_target']:

                    for smooth_a in hyperparams['smoothing_a_vals']:

                        # Creating a good config dict for the runs
                        temp_config = make_temp_config(config_dict=config_dict, w=w, cluster_param=cluster_param, hurdle_model=hurdle_model, 
                                                     smoothing_target=smoothing_target, linear_smooth=True, smooth_a=smooth_a, smooth_k=config_dict['smooth_k'])

                        temp_config_nt = config_nt_class(**temp_config)

                        # Getting the init grids and model
                        _, _, _, u_cluster, v_cluster, p_cluster = get_ll_param_grids(temp_config)
                        model = make_cluster_model(cluster_param=cluster_param, config_dict=temp_config, u_init=u_cluster, v_init=v_cluster, p_init=p_cluster)


                        # Running the LL and getting the output row
                        output_metrics, calibration_outputs, *_ = run_pipeline_ll(model, config_nt=temp_config_nt, train_test_nt=validation_only_nt, config_dict=temp_config, degen_mask=degen_mask)
                        row = make_output_table_row(model=model, output_metrics=output_metrics, config_dict=temp_config, test_valid='valid')

                        results.append(row)

                    for smooth_k in hyperparams['smoothing_k_vals']:

                        # Creating a good config dict for the runs
                        temp_config = make_temp_config(config_dict=config_dict, w=w, cluster_param=cluster_param, hurdle_model=hurdle_model, 
                                                       smoothing_target=smoothing_target, linear_smooth=False, smooth_a=config_dict['smooth_a'], smooth_k=smooth_k)
                        temp_config_nt = config_nt_class(**temp_config)

                        # Getting the init grids and model
                        _, _, _, u_cluster, v_cluster, p_cluster = get_ll_param_grids(temp_config)
                        model = make_cluster_model(cluster_param=cluster_param, config_dict=temp_config, u_init=u_cluster, v_init=v_cluster, p_init=p_cluster)

                        # Running the LL and getting the output row
                        output_metrics, calibration_outputs, *_ = run_pipeline_ll(model, config_nt=temp_config_nt, train_test_nt=validation_only_nt, config_dict=temp_config, degen_mask=degen_mask)
                        row = make_output_table_row(model=model, output_metrics=output_metrics, config_dict=temp_config, test_valid='valid')

                        results.append(row)

    return results

In [ ]:
results = tune_models(hyperparams=hyperparams, train_test_dict=train_test_dict, config_dict=config_dict, degen_mask=degen_mask)

Number of clusters identified : 4


## Creating simple benchmarks for comparison

In [ ]:
def get_user_poisson_rates(user_counts, n_users, train_test_dict, config_dict):

    # Getting the counts per user in the period
    period_df = user_counts.filter((pl.col('fine_bin_id') >= train_test_dict['train_start']) & (pl.col('fine_bin_id') < train_test_dict['burn_in_end']))
    period_df = period_df.group_by('user_id').agg(pl.sum('count').alias('sum_cnt'))

    # Init the poisson rates for each user
    poisson_rates = np.zeros(n_users, dtype='float64')

    # Assinging the mean counts to the users
    users_to_assign = period_df['user_id'].to_numpy()
    poisson_rates[users_to_assign] = period_df['sum_cnt'].to_numpy() / (train_test_dict['burn_in_end'] - train_test_dict['train_start'])
    poisson_rates = np.maximum(poisson_rates, config_dict['mean_min'])

    return poisson_rates

In [ ]:
# Getting the poisson rates
user_poisson_rates = get_user_poisson_rates(user_counts, user_mapping.shape[0], train_test_dict, config_dict)
user_coarse_bin_poisson_rates = u_clustering.copy()

# Creating dictionary and named tuple for with the model index
poisson_benchmark_idx = {'user_poisson' : 0, 'user_coarse_poisson' : 1}
poisson_benchmark_idx_nt_class = dictionary_to_named_tuple_class('poisson_benchmark_idx_nt',poisson_benchmark_idx)
poisson_benchmark_idx_nt = poisson_benchmark_idx_nt_class(**poisson_benchmark_idx)

In [ ]:
@njit
def run_poisson_benchmarks(user_poisson_rates, user_coarse_poisson_rates, degen_mask, 
                           user_counts_nt, user_interactions_nt, train_test_nt, bin_metric_nt, config_nt, poisson_benchmark_idx_nt):
    ''' 
    Runs simple benchmarks one which just fits a poisson model for each user and one which fits a poisson model for each user x coarse bin
    '''
    # Init varaibles and outputs
    n_users = user_poisson_rates.shape[0]
    n_benchmarks = len(poisson_benchmark_idx_nt)
    log_calibration_thresholds = np.log(config_nt.calibration_thresholds)

    # Init outputs
    output_metrics = np.zeros((2, n_benchmarks, 2), dtype='float64')
    calibration_output = np.zeros((2, log_calibration_thresholds.shape[0], n_benchmarks), dtype='float64')

    # Init pointer for user interactions
    usr_frst_rw = _init_user_count_table_pointer(n_users, user_interactions_nt, user_counts_nt, train_test_nt)

    # Iterating over the weeks
    burn_in_first_week = train_test_nt.burn_in_start // bin_metric_nt.fine_bins_per_week
    test_last_week = (train_test_nt.test_end - 1) // bin_metric_nt.fine_bins_per_week

    for week in range(burn_in_first_week, test_last_week + 1):

        week_start = week * bin_metric_nt.fine_bins_per_week
        week_end = (week + 1) * bin_metric_nt.fine_bins_per_week

        if week_end > train_test_nt.test_end:
            week_end = train_test_nt.test_end

        # Iterating over the users and init the pointers
        for user_id in range(n_users):

            cnt_tbl_idx = usr_frst_rw[user_id]
            usr_end_idx = user_interactions_nt.user_last_index[user_id]

            for fine_bin in range(week_start, week_end):

                # Get count and update pointer if necessary get time period and counts coarse bin
                x, cnt_tbl_idx = _get_user_count(cnt_tbl_idx, user_counts_nt, usr_end_idx, fine_bin)

                crnt_coarse_bin, _ = _bin_computations(bin_metric_nt, fine_bin)

                time_period_int = get_time_period(fine_bin, train_test_nt.validation_start, train_test_nt.validation_end, train_test_nt.test_start, train_test_nt.test_end)

                # Compute metrics for non degen bins
                if ((time_period_int == 0 or time_period_int == 1) and not degen_mask[user_id, crnt_coarse_bin]):
                    
                    # Update lambdas and get pmf and upper tail
                    lambda_user = max(user_poisson_rates[user_id], config_nt.mean_min)
                    lambda_user_coarse = max(user_coarse_poisson_rates[user_id, crnt_coarse_bin], config_nt.mean_min,)

                    lpmf_user = poisson_lpmf(x, lambda_user)
                    lpmf_user_coarse = poisson_lpmf(x, lambda_user_coarse)

                    log_upper_tail_user = poisson_log_upper_tail(x, lambda_user)
                    log_upper_tail_user_coarse = poisson_log_upper_tail(x, lambda_user_coarse)

                    # Storing output metrics
                    user_idx = poisson_benchmark_idx_nt.user_poisson
                    user_coarse_idx = poisson_benchmark_idx_nt.user_coarse_poisson

                    output_metrics[time_period_int, user_idx, 0] += 1
                    output_metrics[time_period_int, user_idx, 1] += lpmf_user

                    output_metrics[time_period_int, user_coarse_idx, 0] += 1
                    output_metrics[time_period_int, user_coarse_idx, 1] += lpmf_user_coarse

                    # Storing calibration metrics
                    for calib_threshold_idx in range(log_calibration_thresholds.shape[0]):

                        if log_upper_tail_user < log_calibration_thresholds[calib_threshold_idx]:
                            calibration_output[time_period_int, calib_threshold_idx, user_idx] += 1

                        if log_upper_tail_user_coarse < log_calibration_thresholds[calib_threshold_idx]:
                            calibration_output[time_period_int, calib_threshold_idx, user_coarse_idx] += 1

            usr_frst_rw[user_id] = cnt_tbl_idx

    return output_metrics, calibration_output

In [ ]:
poisson_output_metrics, poisson_calibration_outputs = run_poisson_benchmarks(user_poisson_rates=user_poisson_rates, user_coarse_poisson_rates=user_coarse_bin_poisson_rates, degen_mask=degen_mask, user_counts_nt=user_counts_nt, user_interactions_nt=user_interactions_nt, train_test_nt=train_test_nt, bin_metric_nt=bin_metric_nt, config_nt=config_nt, poisson_benchmark_idx_nt=poisson_benchmark_idx_nt)

In [ ]:
poisson_benchmark_idx = {'user_poisson' : 0, 'user_coarse_poisson' : 1}

def make_poisson_benchmark_output_rows(poisson_output_metrics, test_valid, poisson_benchmark_idx=poisson_benchmark_idx):

    # Gets the period index
    if test_valid == 'valid':
        period_idx = 0
    elif test_valid == 'test':
        period_idx = 1

    # Itertaing over the models
    output = []
    for model_name, model_idx in poisson_benchmark_idx.items():

        # Getting metrics and updating outputs
        n_bins_scored = poisson_output_metrics[period_idx, model_idx, 0]
        ll_sum = poisson_output_metrics[period_idx, model_idx, 1]
        output.append({'smoothed_model_name': model_name, 'test_valid': test_valid, 'non_degen_ll': ll_sum / n_bins_scored})

    return output

In [ ]:
def make_poisson_benchmark_calibration_rows(poisson_output_metrics, poisson_calibration_outputs, config_dict, test_valid, poisson_benchmark_idx=poisson_benchmark_idx):
    
    # Gets the period index
    if test_valid == 'valid':
        period_idx = 0
    elif test_valid == 'test':
        period_idx = 1

    output = []
    for model_name, model_idx in poisson_benchmark_idx.items():

        n_bins_scored = poisson_output_metrics[period_idx, model_idx, 0]

        for threshold_idx in range(config_dict['calibration_thresholds'].shape[0]):
            output.append({'smoothed_model_name': model_name, 
                           'test_valid': test_valid, 
                           'threshold': config_dict['calibration_thresholds'][threshold_idx], 
                           'model_calibration': poisson_calibration_outputs[period_idx, threshold_idx, model_idx] / n_bins_scored})

    return output

### Everything from here onwards is old benchmarks

In [ ]:
# ## Creating 2 named tuple classes to be used in this runner
# ecdf_tbl_class = namedtuple('ecdf', ['user_id', 'fine_bin_id', 'coarse_bin_id', 'count'])
# test_degen_cnts_tbl_class = namedtuple('test_degen_cnts', ['user_id', 'fine_bin_id', 'coarse_bin_id', 'count', 'log_p0_raw'])

# @njit
# def ecdf_benchmark_degen_bins(u_init, v_init,
#                               cluster_u_init, cluster_v_init, cluster_groups,
#                               user_counts_nt, user_interactions_nt,
#                               interpolation_weights,
#                               train_test_nt, bin_metric_nt, config_nt):
#     '''
#     Gets user counts for computing an ECDF 
#     Gets test counts for testing the ECDF model against the test set
#     '''
    
#     u = u_init.copy()
#     v = v_init.copy()

#     cluster_u = cluster_u_init.copy()
#     cluster_v = cluster_v_init.copy()

#     n_users, n_coarse_bins = u.shape

#     test_last_week = (train_test_nt.test_end - 1)// bin_metric_nt.fine_bins_per_week
    
#     degen_bins_mask = np.zeros((n_users, n_coarse_bins), dtype='bool')

#     # Creating ECDF and test observations arrays
#     # Creating max size arrays as we must preallocate fixed size array in numba
#     ecdf_max_rows = n_users * bin_metric_nt.fine_bins_per_week
#     test_degen_cnts_max_rows = n_users * (train_test_nt.test_end - train_test_nt.test_start)

#     ecdf = ecdf_tbl_class(
#         np.zeros(ecdf_max_rows, dtype='int64'),
#         np.zeros(ecdf_max_rows, dtype='int64'),
#         np.zeros(ecdf_max_rows, dtype='int64'),
#         np.zeros(ecdf_max_rows, dtype='int64'))

#     test_degen_cnts = test_degen_cnts_tbl_class(
#         np.zeros(test_degen_cnts_max_rows, dtype='int64'),
#         np.zeros(test_degen_cnts_max_rows, dtype='int64'),
#         np.zeros(test_degen_cnts_max_rows, dtype='int64'),
#         np.zeros(test_degen_cnts_max_rows, dtype='int64'),
#         np.zeros(test_degen_cnts_max_rows, dtype='float64'))


#     # Init user_counts_nt pointer for each user and pointers for which row of the ecdf and test degen cnts tables we are filling in
#     usr_frst_rw = user_interactions_nt.user_first_index.copy()
#     test_degen_row_idx = 0
#     ecdf_row_idx = 0
    
#     for week in range(0, test_last_week + 1):

#         # Getting fine week start and week end
#         week_start = week * bin_metric_nt.fine_bins_per_week
#         week_end = (week + 1) * bin_metric_nt.fine_bins_per_week
        
#         if week_end > train_test_nt.test_end:
#             week_end = train_test_nt.test_end

#         for user_id in range(n_users):
#             # Init pointer and last index for pointer
#             cnt_tbl_idx = usr_frst_rw[user_id]
#             usr_end_idx = user_interactions_nt.user_last_index[user_id]

#             # Init vectors which collect sum of mu values in coarse bins
#             usr_updt_u_sum = np.zeros(n_coarse_bins, dtype='float64')
#             usr_updt_v_sum = np.zeros(n_coarse_bins, dtype='float64')

#             for fine_bin in range(week_start, week_end):

#                 x, cnt_tbl_idx = _get_user_count(cnt_tbl_idx, user_counts_nt, usr_end_idx, fine_bin)

#                 crnt_coarse_bin, crnt_fine_bin_within_coarse_pos = _bin_computations(bin_metric_nt, fine_bin)

#                 # Getting unsmoothed but interpolated params
#                 mu_unsmth_t, sigma_unsmth_2_t = get_smoothed_params(u, v, cluster_u, cluster_v, cluster_groups, 0, user_id, crnt_coarse_bin, 
#                                                     crnt_fine_bin_within_coarse_pos, interpolation_weights)
                
#                 mu_unsmth_t = max(mu_unsmth_t, config_nt.mean_min)
#                 sigma_unsmth_2_t = max(sigma_unsmth_2_t, config_nt.var_min)

#                 log_p0_raw = get_lpmf_val(0, mu_unsmth_t, sigma_unsmth_2_t, config_nt.mean_min, config_nt.var_min, config_nt.min_mean_var_diff)

#                 # Update the mask in the degen identification week
#                 if week == degen_identify_wk:
#                     if log_p0_raw > log_degen_threshold:
#                         degen_bins_mask[user_id, crnt_coarse_bin] = True

#                 # Update the ECDF if we are in the ecdf_wk
#                 if week == ecdf_wk:
#                     if degen_bins_mask[user_id, crnt_coarse_bin]:
#                         ecdf.user_id[ecdf_row_idx] = user_id
#                         ecdf.fine_bin_id[ecdf_row_idx] = fine_bin
#                         ecdf.coarse_bin_id[ecdf_row_idx] = crnt_coarse_bin
#                         ecdf.count[ecdf_row_idx] = x

#                         ecdf_row_idx += 1

#                 if config_nt.use_validation:
#                     recording_period_start = train_test_nt.validation_start
#                     recording_period_end = train_test_nt.validation_end
#                 else: 
#                     recording_period_start = train_test_nt.test_start
#                     recording_period_end = train_test_nt.test_end

#                 if ((fine_bin >= recording_period_start) & (fine_bin < recording_period_end)):
#                     if log_p0_raw > log_degen_threshold:
#                         test_degen_cnts.user_id[test_degen_row_idx] = user_id
#                         test_degen_cnts.fine_bin_id[test_degen_row_idx] = fine_bin
#                         test_degen_cnts.coarse_bin_id[test_degen_row_idx] = crnt_coarse_bin
#                         test_degen_cnts.count[test_degen_row_idx] = x
#                         test_degen_cnts.log_p0_raw[test_degen_row_idx] = log_p0_raw

#                         test_degen_row_idx += 1

#                 collect_temp_grid(usr_updt_u_sum, usr_updt_v_sum, crnt_coarse_bin, x, mu_unsmth_t, sigma_unsmth_2_t, config_nt.w, config_nt.mean_min, config_nt.var_min)

#             usr_frst_rw[user_id] = cnt_tbl_idx

#             update_grid(u, v, user_id, usr_updt_u_sum, usr_updt_v_sum, bin_metric_nt.fine_bins_per_coarse_bin)

#     # Snipping away empty rows in output
#     return (degen_bins_mask, ecdf_tbl_class(ecdf.user_id[:ecdf_row_idx],
#                                             ecdf.fine_bin_id[:ecdf_row_idx],
#                                             ecdf.coarse_bin_id[:ecdf_row_idx],
#                                             ecdf.count[:ecdf_row_idx]),
#                             test_degen_cnts_tbl_class(test_degen_cnts.user_id[:test_degen_row_idx],
#                                                       test_degen_cnts.fine_bin_id[:test_degen_row_idx],
#                                                       test_degen_cnts.coarse_bin_id[:test_degen_row_idx],
#                                                       test_degen_cnts.count[:test_degen_row_idx],
#                                                       test_degen_cnts.log_p0_raw[:test_degen_row_idx]))

In [ ]:
# def get_ecdf_performance_metrics(ecdf_counts, test_degen):


#     non_zero_ecdf_counts = ecdf_counts.count[ecdf_counts.count > 0]
#     output = {'ecdf_n_degen_fine_bins' : len(ecdf_counts.count),
#               'ecdf_p_non_zero' : len(non_zero_ecdf_counts)/ecdf_counts,
#               'ecdf_non_zero_mean' : non_zero_ecdf_counts.mean(),
#               'ecdf_non_zero_var' : non_zero_ecdf_counts.var(),
#               }
    
#     # Getting the ECDF mean prediction and a vector of this prediction.
#     output['ecdf_mean'] = output['ecdf_p_non_zero'] * output['ecdf_non_zero_mean']
#     ecdf_pred = np.ones(len(test_degen.count), dtype='float64') * output['ecdf_mean']

#     user_maes = []

#     for user_id in np.unique(test_degen.user_id):
#         crnt_usr_msk = user_id == test_degen.user_id
#         user_maes.append(np.abs(test_degen.count[crnt_usr_msk] - ecdf_pred[crnt_usr_msk]).mean())
    
#     output['ecdf_user_mean_mae'] = user_maes.mean()
#     output['ecdf_mae'] = np.abs(np.abs(test_degen.count - ecdf_pred).mean())
    
#     output['n_test_degen_rows'] = len(test_degen.count)
#     return output

The benchmark we will use is an empirical CDF which looks at the counts 1 hour either side of the data point in the previous n weeks and then uses that to make an ECDF for the model. It will be used to judge the non degen bins LL

In [ ]:
# @njit
# def get_simple_benchmark_performance(cnts_tbl_f_bn_id, cnts_tbl_cnt, intract_tbl_frst_int, intract_tbl_lst_int, validation_start, validation_end, 
#                                      test_start, test_end, fine_bins_per_week, window_radius_fb, historical_weeks_used):
#     ''' 
#     Runs a simple benchmark model which compares counts against an empirical CDF of observed historical counts in that bin and the adjacent bins
#     window_radius_fb - number of adjacent fine bins to used for the comparison
#     historical_weeks_used - the number of historical weeks used for the comparison
#     '''
#     # Get users to loop over and init output
#     n_users = intract_tbl_frst_int.shape[0]
#     output = np.zeros((2,4), dtype=np.float64)
    
#     for user_id in range(n_users):

#         # Getting rows to iterate over in counts df for that user
#         user_first_row = intract_tbl_frst_int[user_id]
#         user_last_row = intract_tbl_lst_int[user_id]

#         ## Building an array of user counts including 0 rows lenght is test end (fine bins in dataset)
#         user_counts = np.zeros(test_end, dtype='int64')
#         for row in range(user_first_row, user_last_row + 1):
#             fine_bin_id = cnts_tbl_f_bn_id[row]
#             if fine_bin_id < test_end:
#                 user_counts[fine_bin_id] = cnts_tbl_cnt[row]

#         for fine_bin_id in range(validation_start, test_end):
#             period_idx = get_time_period(fine_bin_id, validation_start, validation_end, test_start, test_end)

#             if period_idx == -1:
#                 continue

#             # Getting the count for comparison and initalising outputs
#             comparison_count = user_counts[fine_bin_id]
#             observed_greater_or_equal_to_count = 0
#             number_of_bins = 0
#             observed_equal_to_count = 0

#             # looping over historical fine bins and finding the historical observed count using our array
#             for weeks_ago  in range(1, historical_weeks_used + 1):
#                 central_fine_bin = fine_bin_id - weeks_ago * fine_bins_per_week
#                 for fine_bin_offset in range(-window_radius_fb, window_radius_fb + 1):
#                     historical_fine_bin = central_fine_bin + fine_bin_offset
#                     assert historical_fine_bin >= 0
#                     historical_observed_count = user_counts[historical_fine_bin]

#                     # Calculating the upper tail probability and updating the output metrics
#                     number_of_bins += 1
#                     if historical_observed_count >= comparison_count:
#                         observed_greater_or_equal_to_count +=1
#                     if historical_observed_count == comparison_count:
#                         observed_equal_to_count += 1

#             upper_tail_prob = observed_greater_or_equal_to_count / number_of_bins
#             strict_upper_tail_prob = (observed_greater_or_equal_to_count - observed_equal_to_count) / number_of_bins

#             output[period_idx, 0] += observed_greater_or_equal_to_count
#             output[period_idx, 1] += upper_tail_prob
#             output[period_idx, 2] += strict_upper_tail_prob
#             output[period_idx, 3] += 1

#     return output